# 03 - RAG Pipeline Evaluation

End-to-end evaluation of the FinanceRAG pipeline using [RAGAS](https://github.com/explodinggradients/ragas).

Metrics evaluated:
- **Faithfulness**: Is the answer grounded in the retrieved context?
- **Answer Relevancy**: Is the answer relevant to the question?
- **Context Precision**: Are the retrieved chunks relevant?
- **Context Recall**: Were all necessary chunks retrieved?

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

plt.style.use('seaborn-v0_8-whitegrid')

## 1. Define Evaluation Dataset

Create a set of question-answer pairs with expected source sections.
These serve as our ground truth for evaluating the RAG pipeline.

In [2]:
# Evaluation dataset: questions with expected answers and source sections
eval_dataset = [
    {
        'question': 'What are Apple\'s main risk factors related to global economic conditions?',
        'expected_section': 'Item 1A',
        'expected_ticker': 'AAPL',
        'reference_answer': 'Apple faces risks from adverse global and regional economic conditions, including inflation, slower growth, and recession, which could cause consumers to defer purchases.',
    },
    {
        'question': 'What was Apple\'s total revenue for fiscal year 2024?',
        'expected_section': 'Item 7',
        'expected_ticker': 'AAPL',
        'reference_answer': 'Revenue was $383.3 billion for fiscal year 2024, compared to $394.3 billion for fiscal year 2023.',
    },
    {
        'question': 'How did Apple\'s services revenue change year over year?',
        'expected_section': 'Item 7',
        'expected_ticker': 'AAPL',
        'reference_answer': 'Services revenue was $85.2 billion, an increase of 13% compared to fiscal year 2023.',
    },
    {
        'question': 'What products does Apple manufacture and sell?',
        'expected_section': 'Item 1',
        'expected_ticker': 'AAPL',
        'reference_answer': 'Apple designs, manufactures, and markets smartphones, personal computers, tablets, wearables, and accessories. Products include iPhone, Mac, iPad, and Wearables.',
    },
    {
        'question': 'What was Apple\'s gross margin for fiscal year 2024 and how did it compare to 2023?',
        'expected_section': 'Item 7',
        'expected_ticker': 'AAPL',
        'reference_answer': 'Gross margin was 46.2% for fiscal year 2024, compared to 44.1% for fiscal year 2023, driven by a shift toward higher-margin Services.',
    },
]

df_eval = pd.DataFrame(eval_dataset)
print(f'Evaluation dataset: {len(df_eval)} questions')
df_eval[['question', 'expected_section']].head()

ArrowTypeError: Input object was not a NumPy array

## 2. Run Pipeline on Evaluation Questions

Query the RAG pipeline for each evaluation question and collect:
- Generated answer
- Retrieved contexts
- Source metadata

In [ ]:
import requests

API_URL = 'http://localhost:8000/api/query'

results = []
for _, row in df_eval.iterrows():
    try:
        resp = requests.post(API_URL, json={
            'question': row['question'],
            'filters': {'ticker': row['expected_ticker']},
            'top_k': 5,
        }, timeout=60)
        data = resp.json()
        results.append({
            'question': row['question'],
            'reference_answer': row['reference_answer'],
            'generated_answer': data.get('answer', ''),
            'sources': data.get('sources', []),
            'expected_section': row['expected_section'],
            'guardrail_warnings': data.get('guardrail_warnings', []),
        })
        print(f'✓ {row["question"][:60]}...')
    except Exception as e:
        print(f'✗ Failed: {e}')
        results.append({
            'question': row['question'],
            'reference_answer': row['reference_answer'],
            'generated_answer': f'ERROR: {e}',
            'sources': [],
            'expected_section': row['expected_section'],
            'guardrail_warnings': [],
        })

df_results = pd.DataFrame(results)
print(f'\nCompleted {len(df_results)} evaluations')

## 3. Retrieval Accuracy

Check if the correct section was retrieved for each question.

In [ ]:
# Check if expected section appears in retrieved sources
def check_section_retrieved(row):
    for source in row['sources']:
        if source.get('section', '').startswith(row['expected_section']):
            return True
    return False

df_results['correct_section_retrieved'] = df_results.apply(check_section_retrieved, axis=1)

retrieval_accuracy = df_results['correct_section_retrieved'].mean()
print(f'Section Retrieval Accuracy: {retrieval_accuracy:.0%}')
print()
for _, row in df_results.iterrows():
    status = '✓' if row['correct_section_retrieved'] else '✗'
    retrieved_sections = [s.get('section', 'N/A') for s in row['sources'][:3]]
    print(f'  {status} Expected: {row["expected_section"]} | Retrieved: {retrieved_sections}')

## 4. RAGAS Evaluation

Run RAGAS metrics for systematic evaluation.

In [ ]:
# NOTE: Uncomment and run when the pipeline is fully operational
# from ragas import evaluate
# from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
# from datasets import Dataset
#
# # Prepare RAGAS dataset format
# ragas_data = {
#     'question': df_results['question'].tolist(),
#     'answer': df_results['generated_answer'].tolist(),
#     'contexts': [
#         [s.get('text', '') for s in row['sources']]
#         for _, row in df_results.iterrows()
#     ],
#     'ground_truth': df_results['reference_answer'].tolist(),
# }
#
# dataset = Dataset.from_dict(ragas_data)
# result = evaluate(
#     dataset,
#     metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
# )
# print(result)

# Placeholder results for demonstration
print('RAGAS Evaluation (run with live pipeline):')
print('  Faithfulness:       TBD')
print('  Answer Relevancy:   TBD')
print('  Context Precision:  TBD')
print('  Context Recall:     TBD')

## 5. Guardrail Effectiveness

Verify that the guardrail module catches issues appropriately.

In [ ]:
from modules.generation.guardrails import ResponseGuardrails

guardrails = ResponseGuardrails()

# Test cases for guardrails
test_cases = [
    {
        'name': 'Good response with citation',
        'response': 'Apple\'s revenue was $383.3 billion [Source: AAPL, 10-K, 2024, Item 7].',
        'should_pass': True,
    },
    {
        'name': 'Missing citations',
        'response': 'Apple\'s revenue decreased due to lower iPhone sales.',
        'should_pass': True,  # passes but with warnings
    },
    {
        'name': 'Investment advice',
        'response': 'Based on this analysis, you should buy AAPL stock.',
        'should_pass': False,
    },
    {
        'name': 'Hallucinated number',
        'response': 'Revenue was $999.9 billion [Source: AAPL, 10-K, 2024, Item 7].',
        'should_pass': True,  # passes but with warnings
    },
]

sample_sources = [{'text': 'Revenue was $383.3 billion for fiscal year 2024.'}]

print('Guardrail Test Results:')
print(f'{"Test Case":<30} {"Passed":<10} {"Warnings":<10} {"Flags":<10} {"Expected"}')
print('-' * 80)

for tc in test_cases:
    result = guardrails.check_all(tc['response'], sample_sources)
    status = '✓' if result.passed == tc['should_pass'] else '✗'
    print(
        f'{status} {tc["name"]:<28} '
        f'{str(result.passed):<10} '
        f'{len(result.warnings):<10} '
        f'{len(result.flags):<10} '
        f'pass={tc["should_pass"]}'
    )

## 6. Results Visualization

In [ ]:
# Visualization placeholder — update with real RAGAS scores
metrics = ['Faithfulness', 'Answer Relevancy', 'Context Precision', 'Context Recall']
# Replace with actual scores from RAGAS evaluation
scores = [0.0, 0.0, 0.0, 0.0]  # TBD

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12']
bars = ax.barh(metrics, scores, color=colors, edgecolor='black', alpha=0.8)
ax.set_xlim(0, 1.0)
ax.set_xlabel('Score')
ax.set_title('RAG Pipeline Evaluation Metrics (RAGAS)')

for bar, score in zip(bars, scores):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
            f'{score:.2f}', va='center', fontweight='bold')

plt.tight_layout()
plt.show()

print('\nNote: Run the full pipeline and update scores above with real RAGAS results.')

## Summary

### Key Findings

| Component | Decision | Impact |
|---|---|---|
| Chunking | Section-based + Recursive | Preserves section boundaries, enables metadata filtering |
| Retrieval | Hybrid (Dense + BM25) + Reranking | Captures both semantic and keyword matches |
| Guardrails | Citation + advice + hallucination checks | Ensures factual grounding and compliance |

### Next Steps
1. Run RAGAS evaluation with live pipeline
2. Fine-tune embedding model on financial corpus
3. A/B test different chunk sizes (256 vs 512 vs 1024)
4. Add multi-document comparison evaluation